In [27]:
import torch
import torch.nn as nn
from torch.functional import F
import torchvision
import math

In [28]:
torch_tensor = torch.tensor([[1, 2, 3],
                             [4, 5, 6]],
                            dtype=torch.float,
                            device="cpu",
                            requires_grad=True
)
torch_tensor

tensor([[1., 2., 3.],
        [4., 5., 6.]], requires_grad=True)

In [29]:
x = torch.clone(torch_tensor)
y = torch.add(x, torch_tensor)
y

tensor([[ 2.,  4.,  6.],
        [ 8., 10., 12.]], grad_fn=<AddBackward0>)

In [30]:
def hubber_fn(y_true, y_pred):
  error = y_true - y_pred
  abs_error = torch.abs(error) < 1
  square_loss = torch.square(error) / 2
  linear_loss = torch.abs(error)
  return torch.where(abs_error, square_loss, linear_loss)

In [31]:
y_true = torch.tensor([1, 2, 3])
y_pred = torch.tensor([1, 3, 3])

hubber_fn(y_true, y_pred)

tensor([0., 1., 0.])

In [32]:
def create_hubber(delta=1.0):
  def hubber_fn(y_true, y_pred):
    error = y_true - y_pred
    is_small_error = torch.abs(error) < delta
    squared_loss = torch.square(error) / 2
    linear_loss = delta * torch.abs(error) - delta ** 2 / 2
    return torch.where(is_small_error, squared_loss, linear_loss)
  return hubber_fn

In [33]:
class HuberLoss(nn.Module):
    def __init__(self, delta=1.0):
        super().__init__()
        self.delta = delta

    def forward(self, y_true, y_pred):
        error = y_true - y_pred
        abs_error = torch.abs(error)
        is_small_error = abs_error < self.delta
        squared_loss = 0.5 * (error ** 2)
        linear_loss = self.delta * abs_error - 0.5 * (self.delta ** 2)
        return torch.where(is_small_error, squared_loss, linear_loss).mean()

In [34]:
class MyL1Regularizer(torch.nn.Module):
    def __init__(self, factor):
        super().__init__()
        self.factor = factor

    def forward(self, weight):
        return self.factor * torch.sum(torch.abs(weight))

In [35]:
reg = MyL1Regularizer(0.01)
w = torch.randn(10, 10)
penalty = reg(w)
print(penalty)

tensor(0.9044)


In [36]:
criterion = HuberLoss(delta=1.0)
loss = criterion(y_true, y_pred)
loss

tensor(0.1667)

In [37]:
class HuberMetric(nn.Module):
    def __init__(self, delta=1.0):
        super().__init__()
        self.delta = delta
        self.total = 0.0
        self.count = 0
        self.huber_fn = self._huber_loss_fn

    def _huber_loss_fn(self, y_true, y_pred):
        error = y_true - y_pred
        abs_error = torch.abs(error)
        is_small_error = abs_error < self.delta
        squared_loss = 0.5 * (error ** 2)
        linear_loss = self.delta * abs_error - 0.5 * (self.delta ** 2)
        return torch.where(is_small_error, squared_loss, linear_loss)

    def update_state(self, y_true, y_pred, sample_weight=None):
        loss = self.huber_fn(y_true, y_pred)
        if sample_weight is not None:
            loss = loss * sample_weight
        self.total += loss.sum().item()
        self.count += y_true.numel()

    def result(self):
        if self.count == 0:
            return 0.0
        return self.total / self.count

    def reset(self):
        self.total = 0.0
        self.count = 0

In [38]:
exponential_layer = torchvision.transforms.Lambda(lambda x: torch.exp(torch.tensor(x, dtype=torch.float32)))

exponential_layer([5, 3])

tensor([148.4132,  20.0855])

In [39]:
class MyDense(nn.Module):
    def __init__(self, in_features, out_features, bias=True):
        super().__init__()
        # Inicializa pesos e bias como parâmetros treináveis
        self.weight = nn.Parameter(torch.randn(out_features, in_features))
        if bias:
            self.bias = nn.Parameter(torch.zeros(out_features))
        else:
            self.bias = None

    def forward(self, x):
        # x: (batch_size, in_features)
        # output: (batch_size, out_features)
        output = x @ self.weight.T
        if self.bias is not None:
            output += self.bias
        return output

In [40]:
my_dense = MyDense(8, 4)
print(f"Pesos iniciais:\n {my_dense.weight}")
print(f"\nBias iniciais:\n {my_dense.bias}")

Pesos iniciais:
 Parameter containing:
tensor([[-0.0954,  0.4728,  1.2689,  0.9843, -0.7514,  0.0534, -2.0700,  0.3546],
        [ 0.1449, -0.1083,  0.2148, -1.0048,  1.1292,  0.3599,  0.4912,  0.3537],
        [ 0.5607,  0.8568, -0.2138,  0.6328,  0.1381, -0.1892,  0.7613,  0.1438],
        [ 2.0273, -0.0661,  1.2422,  1.1379, -0.5256,  0.3805, -1.9180, -1.4736]],
       requires_grad=True)

Bias iniciais:
 Parameter containing:
tensor([0., 0., 0., 0.], requires_grad=True)


In [41]:
x = torch.randn(10, 8)

out = my_dense(x)
out.shape

torch.Size([10, 4])

In [42]:
class MultiLayer(nn.Module):
  def __init__(self):
    super().__init__()

  def forward(self, X):
    x1, x2 = X
    return [x1+x2, x1*x2, x1/x2]

In [43]:
layer = MultiLayer()

x1 = torch.tensor([1.0, 2.0, 3.0])
x2 = torch.tensor([4.0, 0.5, 2.0])

out = layer((x1, x2))
print("Soma:", out[0])
print("Produto:", out[1])
print("Divisão:", out[2])

Soma: tensor([5.0000, 2.5000, 5.0000])
Produto: tensor([4., 1., 6.])
Divisão: tensor([0.2500, 4.0000, 1.5000])


In [44]:
class MyGaussianNoise(nn.Module):
    def __init__(self, stddev):
        super().__init__()
        self.stddev = stddev

    def forward(self, x):
        # Só adiciona ruído durante o treinamento
        if self.training:
            noise = torch.randn_like(x) * self.stddev
            return x + noise
        else:
            return x

In [45]:
layer = MyGaussianNoise(0.1)

x = torch.ones(3)

layer.train()
print("Treinamento:", layer(x))

layer.eval()
print("Avaliação:", layer(x))

Treinamento: tensor([0.9602, 0.8258, 0.9405])
Avaliação: tensor([1., 1., 1.])


In [46]:
class ResidualBlock(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()

        # Primeira camada linear
        self.fc1 = nn.Linear(in_features, out_features)
        # Segunda camada linear
        self.fc2 = nn.Linear(out_features, out_features)

        # Caso as dimensões não batam, cria um atalho para ajustar
        if in_features != out_features:
            self.shortcut = nn.Linear(in_features, out_features)
        else:
            self.shortcut = nn.Identity()

        # Normalização e ativação
        self.bn1 = nn.BatchNorm1d(out_features)
        self.bn2 = nn.BatchNorm1d(out_features)
        self.relu = nn.ReLU()

    def forward(self, x):
        identity = self.shortcut(x)

        out = self.fc1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.fc2(out)
        out = self.bn2(out)

        out += identity
        out = self.relu(out)
        return out

In [47]:
class ResidualRegressor(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.hidden = nn.Linear(input_dim, hidden_dim)

        self.block1 = ResidualBlock(hidden_dim, hidden_dim)
        self.block2 = ResidualBlock(hidden_dim, hidden_dim)

        self.output = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = F.relu(self.hidden(x))
        x = self.block1(x)
        x = self.block2(x)
        x = self.output(x)
        return x

In [49]:
class ReconstructingRegressor(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(ReconstructingRegressor, self).__init__()

        # Lista para armazenar as 5 camadas ocultas
        self.hidden = nn.ModuleList()

        # Adiciona a primeira camada oculta (input_dim -> 30)
        self.hidden.append(nn.Linear(input_dim, 30))

        # Adiciona as 4 camadas ocultas restantes (30 -> 30)
        for _ in range(4):
            self.hidden.append(nn.Linear(30, 30))

        self.out = nn.Linear(30, output_dim)

        # Camada de reconstrução
        self.reconstruct = nn.Linear(30, input_dim)

        # Inicialização dos pesos (lecun_normal)
        for layer in self.hidden:
            # std=sqrt(1/fan_in) é a inicialização 'lecun_normal'
            std_dev = math.sqrt(1.0 / layer.in_features)
            nn.init.normal_(layer.weight, mean=0.0, std=std_dev)
            if layer.bias is not None:
                nn.init.zeros_(layer.bias)

    def forward(self, inputs):

        Z = inputs

        # Passa pelos 5 layers ocultos com ativação SELU
        for layer in self.hidden:
            Z = F.selu(layer(Z))

        # Saída principal da regressão
        main_output = self.out(Z)

        # Caminho da reconstrução
        reconstruction = self.reconstruct(Z)

        # Cálculo da perda de reconstrução
        recon_loss = F.mse_loss(reconstruction, inputs)

        return main_output, recon_loss